In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
from collections import deque, namedtuple
import math 

np.bool8 = np.bool_

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [21]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'reward', 'next_state', 'done'))

In [22]:
class NoisyLinear(nn.Module):
    def __init__(self, in_f, out_f, sigma_0=0.5):
        super().__init__()
        self.in_f, self.out_f = in_f, out_f
        self.sigma_0 = sigma_0

        # NOTE:  (out_f, in_f)  — đúng chiều cho F.linear
        self.weight_mu    = nn.Parameter(torch.empty(out_f, in_f))
        self.weight_sigma = nn.Parameter(torch.empty(out_f, in_f))
        self.bias_mu      = nn.Parameter(torch.empty(out_f))
        self.bias_sigma   = nn.Parameter(torch.empty(out_f))

        self.register_buffer('weight_eps', torch.zeros(out_f, in_f))
        self.register_buffer('bias_eps',   torch.zeros(out_f))

        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_f)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.bias_mu.data.uniform_(-mu_range, mu_range)

        self.weight_sigma.data.fill_(self.sigma_0 / math.sqrt(self.in_f))
        self.bias_sigma.data.fill_(self.sigma_0 / math.sqrt(self.out_f))

    @staticmethod
    def _f(x):
        return torch.sign(x) * torch.sqrt(torch.abs(x) + 1e-10)

    def reset_noise(self):
        eps_in  = self._f(torch.randn(self.in_f, device=device))
        eps_out = self._f(torch.randn(self.out_f, device=device))
        # factorised gaussian
        self.weight_eps.copy_(eps_out.unsqueeze(1) @ eps_in.unsqueeze(0))
        self.bias_eps.copy_(eps_out)

    def forward(self, x):
        if self.training:
            weight = self.weight_mu + self.weight_sigma * self.weight_eps
            bias   = self.bias_mu   + self.bias_sigma   * self.bias_eps
        else:
            weight, bias = self.weight_mu, self.bias_mu
        return torch.nn.functional.linear(x, weight, bias)


In [23]:
# --- Dueling C51 Network with NoisyNet ---
class RainbowNet(nn.Module):
    def __init__(self, obs_dim, n_actions, n_atoms):
        super().__init__()
        self.n_atoms = n_atoms
        self.n_actions = n_actions

        self.feature = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
        )

        # Dueling streams
        self.value_stream = nn.Sequential(
            NoisyLinear(128, 128),
            nn.ReLU(),
            NoisyLinear(128, n_atoms)
        )
        self.advantage_stream = nn.Sequential(
            NoisyLinear(128, 128),
            nn.ReLU(),
            NoisyLinear(128, n_actions * n_atoms)
        )

    def forward(self, x):
        batch_size = x.size(0)
        features = self.feature(x)

        value = self.value_stream(features).view(batch_size, 1, self.n_atoms)
        adv = self.advantage_stream(features).view(batch_size, self.n_actions, self.n_atoms)

        q_atoms = value + adv - adv.mean(dim=1, keepdim=True)
        return q_atoms

    def reset_noise(self):
        for m in self.modules():
            if isinstance(m, NoisyLinear):
                m.reset_noise()

In [24]:
# --- Experience Replay with Prioritized + n-step ---
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity, self.alpha = capacity, alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)

    def push(self, transition: Transition, priority=1.0):
        max_prio = self.priorities.max() if self.buffer else priority
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs  = prios ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        total   = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights = torch.tensor(weights / weights.max(), device=device, dtype=torch.float32)

        batch = Transition(*zip(*samples))
        return batch, indices, weights

    def update_priorities(self, indices, priorities):
        for idx, prio in zip(indices, priorities):
            self.priorities[idx] = prio

    def __len__(self):
        return len(self.buffer)

In [25]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity, self.alpha = capacity, alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)

    def push(self, transition: Transition, priority=1.0):
        max_prio = self.priorities.max() if self.buffer else priority
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs  = prios ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        total   = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights = torch.tensor(weights / weights.max(), device=device, dtype=torch.float32)

        batch = Transition(*zip(*samples))
        return batch, indices, weights

    def update_priorities(self, indices, priorities):
        for idx, prio in zip(indices, priorities):
            self.priorities[idx] = prio

    def __len__(self):
        return len(self.buffer)

In [26]:
class RainbowAgent:
    def __init__(self,
                 obs_dim,
                 n_actions,
                 n_atoms=51, V_min=-200, V_max=200,
                 gamma=0.99,
                 n_step=3,
                 lr=1e-4,
                 buffer_size=100_000,
                 batch_size=64,
                 target_sync=2000,
                 alpha=0.6, beta_start=0.4, beta_frames=500_000):

        self.n_actions = n_actions
        self.n_atoms   = n_atoms
        self.V_min, self.V_max = V_min, V_max
        self.delta_z = (V_max - V_min) / (n_atoms - 1)
        self.z = torch.linspace(V_min, V_max, n_atoms, device=device)

        self.gamma, self.n_step = gamma, n_step
        self.batch_size, self.target_sync = batch_size, target_sync

        self.online  = RainbowNet(obs_dim, n_actions, n_atoms).to(device)
        self.target  = RainbowNet(obs_dim, n_actions, n_atoms).to(device)
        self.target.load_state_dict(self.online.state_dict())

        self.optimizer = optim.Adam(self.online.parameters(), lr=lr)

        self.memory = PrioritizedReplayBuffer(buffer_size, alpha=alpha)
        self.nstep_buffer = deque(maxlen=n_step)

        self.learn_steps = 0
        self.beta_start, self.beta_frames = beta_start, beta_frames
        self.frame_idx = 0

    # --------------------------------------------------
    #  n-step helper
    # --------------------------------------------------
    def _calc_n_step_return(self):
        R, next_state, done = 0.0, None, False
        for idx, (_, _, r, n_s, d) in enumerate(self.nstep_buffer):
            R += (self.gamma ** idx) * r
            next_state, done = n_s, d
            if d:
                break
        state, action, _, _, _ = self.nstep_buffer[0]
        return state, action, R, next_state, done

    # --------------------------------------------------
    #  Interaction
    # --------------------------------------------------
    def act(self, state):
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits = self.online(state)
            probs  = torch.softmax(logits, dim=2)          # (1,A,N)
            qvals  = torch.sum(probs * self.z, dim=2)       # (1,A)
            action = qvals.argmax(1).item()
        return action

    def remember(self, state, action, reward, next_state, done):
        self.nstep_buffer.append(Transition(state, action, reward, next_state, done))
        if len(self.nstep_buffer) < self.n_step:
            return
        n_state, n_action, n_R, n_next, n_done = self._calc_n_step_return()
        self.memory.push(Transition(n_state, n_action, n_R, n_next, n_done))

        if done:                            # clear partial episode
            self.nstep_buffer.clear()

    # --------------------------------------------------
    #  Distribution projection (C51)
    # --------------------------------------------------
    def _l2_project(self, target_dist, rewards, dones):
        batch_size = rewards.size(0)
        proj_dist = torch.zeros(batch_size, self.n_atoms, device=device)

        Tz = rewards.unsqueeze(1) + (1 - dones.unsqueeze(1)) * (self.gamma ** self.n_step) * self.z
        Tz = Tz.clamp(self.V_min, self.V_max)
        b  = (Tz - self.V_min) / self.delta_z
        l  = b.floor().long()
        u  = b.ceil().long()

        offset = torch.linspace(0, (batch_size - 1) * self.n_atoms,
                                batch_size, device=device).long().unsqueeze(1)

        proj_dist.view(-1).index_add_(0, (l + offset).view(-1),
            (target_dist * (u.float() - b)).view(-1))
        proj_dist.view(-1).index_add_(0, (u + offset).view(-1),
            (target_dist * (b - l.float())).view(-1))

        proj_dist = proj_dist + 1e-8         # avoid zero
        proj_dist /= proj_dist.sum(dim=1, keepdim=True)
        return proj_dist

    # --------------------------------------------------
    #  Training
    # --------------------------------------------------
    def beta_by_frame(self, frame_idx):
        return min(1.0, self.beta_start + (1.0 - self.beta_start) * frame_idx / self.beta_frames)

    def train_step(self):
        if len(self.memory) < 10_000:
            return

        # 🟢 Reset noise ONCE – trước mọi forward
        self.online.reset_noise()
        self.target.reset_noise()

        beta = self.beta_by_frame(self.frame_idx)
        batch, idxs, weights = self.memory.sample(self.batch_size, beta=beta)

        # ---------- unpack tensors ----------
        states  = torch.tensor(batch.state, dtype=torch.float32, device=device)
        actions = torch.tensor(batch.action, dtype=torch.int64,  device=device)
        rewards = torch.tensor(batch.reward, dtype=torch.float32, device=device)
        next_s  = torch.tensor(batch.next_state, dtype=torch.float32, device=device)
        dones   = torch.tensor(batch.done, dtype=torch.float32, device=device)

        # ---------- online forward ----------
        logits = self.online(states)                        # (B,A,N)
        log_p  = torch.log_softmax(logits, dim=2)
        chosen_log_p = log_p[range(self.batch_size), actions]

        # ---------- target distribution ----------
        with torch.no_grad():
            next_logits  = self.online(next_s)              # uses same noise sample
            next_probs   = torch.softmax(next_logits, dim=2)
            q_next       = torch.sum(next_probs * self.z, dim=2)
            next_actions = q_next.argmax(1)

            target_logits = self.target(next_s)
            target_probs  = torch.softmax(target_logits, dim=2)
            next_dist     = target_probs[range(self.batch_size), next_actions]

            proj_dist = self._l2_project(next_dist, rewards, dones)

        # ---------- loss & update ----------
        loss = -(proj_dist * chosen_log_p).sum(1)
        loss = (loss * weights).mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), 10.0)
        self.optimizer.step()

        # ---------- priority update ----------
        with torch.no_grad():
            td_err = torch.sum(torch.abs(proj_dist - torch.exp(chosen_log_p)), dim=1)
        self.memory.update_priorities(idxs, td_err.cpu().numpy() + 1e-6)

        # ---------- target sync ----------
        self.learn_steps += 1
        if self.learn_steps % self.target_sync == 0:
            self.target.load_state_dict(self.online.state_dict())


In [ ]:
def train_rainbow(env_name="LunarLander-v2",
                  episodes=3000,
                  max_steps=500):
    env = gym.make(env_name)
    agent = RainbowAgent(obs_dim=env.observation_space.shape[0],
                         n_actions=env.action_space.n)

    scores, avgs = [], []
    for ep in range(episodes):
        state, _ = env.reset()
        total_r = 0
        for t in range(max_steps):
            agent.frame_idx += 1
            action = agent.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.remember(state, action, reward, next_state, done)
            agent.train_step()

            state = next_state
            total_r += reward
            if done:
                break

        scores.append(total_r)
        avgs.append(np.mean(scores[-50:]))
        print(f"Ep {ep:4d} | R {total_r:7.2f} | Avg50 {avgs[-1]:7.2f}")

        if avgs[-1] >= 200 and ep >= 50:
            print("Solved environment!")
            break

    env.close()
    # Simple plot
    plt.plot(scores, label='Reward')
    plt.plot(avgs, label='Moving Avg (50)')
    plt.legend(); plt.grid(); plt.show()

# --------------------------------------------------
if __name__ == "__main__":
    import gym, matplotlib.pyplot as plt, numpy as np
    train_rainbow()

Ep    0 | R -534.79 | Avg50 -534.79
Ep    1 | R -640.69 | Avg50 -587.74
Ep    2 | R -406.79 | Avg50 -527.42
Ep    3 | R -920.41 | Avg50 -625.67
Ep    4 | R -582.80 | Avg50 -617.10
Ep    5 | R -554.31 | Avg50 -606.63
Ep    6 | R -564.23 | Avg50 -600.57
Ep    7 | R -599.21 | Avg50 -600.40
Ep    8 | R -454.95 | Avg50 -584.24
Ep    9 | R -1008.18 | Avg50 -626.64
Ep   10 | R -486.03 | Avg50 -613.85
Ep   11 | R -632.89 | Avg50 -615.44
Ep   12 | R -1684.71 | Avg50 -697.69
Ep   13 | R -538.38 | Avg50 -686.31
Ep   14 | R -661.31 | Avg50 -684.65
Ep   15 | R -1256.41 | Avg50 -720.38
Ep   16 | R -570.04 | Avg50 -711.54
Ep   17 | R -569.52 | Avg50 -703.65
Ep   18 | R -840.20 | Avg50 -710.83
Ep   19 | R -740.64 | Avg50 -712.32
Ep   20 | R -632.00 | Avg50 -708.50
Ep   21 | R -947.79 | Avg50 -719.38
Ep   22 | R -1019.13 | Avg50 -732.41
Ep   23 | R -587.64 | Avg50 -726.38
Ep   24 | R -715.93 | Avg50 -725.96
Ep   25 | R -626.49 | Avg50 -722.13
Ep   26 | R -706.42 | Avg50 -721.55
Ep   27 | R -589.16 | Av